In [8]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

print(PROJECT_ROOT)

c:\Users\Nahid\dmpchef


In [9]:
import yaml

CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"

print("Config path:", CONFIG_PATH)

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

print("Config loaded")

Config path: c:\Users\Nahid\dmpchef\config\config.yaml
Config loaded


In [10]:
INDEX_DIR = PROJECT_ROOT / cfg["paths"]["index_dir"]

print("Vector DB path:", INDEX_DIR)

Vector DB path: c:\Users\Nahid\dmpchef\data\vector_db\DMPtools_db


In [11]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name=cfg["models"]["embedding_model"],
    model_kwargs={"device": cfg["models"]["embedding_device"]}
)

vs = FAISS.load_local(
    str(INDEX_DIR),
    embeddings,
    allow_dangerous_deserialization=True
)

print("FAISS loaded")
print("Total vectors:", vs.index.ntotal)

C:\Users\Nahid\AppData\Local\Temp\ipykernel_41104\104530311.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
c:\Users\Nahid\dmpchef\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1680.95it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------

FAISS loaded
Total vectors: 1784


In [17]:
def keyword_search(vs, keyword):
    
    results = []

    for doc_id in vs.index_to_docstore_id.values():

        doc = vs.docstore.search(doc_id)

        if keyword.lower() in doc.page_content.lower():

            results.append(doc)

    return results


hits = keyword_search(vs, "labname")

print("Matches:", len(hits))

for h in hits[:5]:
    print(h.metadata)
    print(h.page_content[:300])
    print("------")

Matches: 0


In [18]:
keyword_search(vs, "labname")


[]

In [19]:
keyword_search(vs, "University X")

[]

In [20]:
# ============================================
# STEP 1 — Imports, Config (YAML), and Helpers
# ============================================
import os, re, time
from pathlib import Path
from datetime import datetime

import yaml
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
import pypandoc  # for Markdown → DOCX

# --- LangChain Core ---
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()

# ---------- Resolve project root (works in notebook or script) ----------
try:
    PROJECT_ROOT = Path(__file__).resolve().parents[1]  
except NameError:
    PROJECT_ROOT = Path.cwd().parent                   

# ---------- Notebook-only output root (keeps experiments separate) ----------
NOTEBOOK_DIR = Path.cwd()  # folder where the notebook is running
NB_OUT_ROOT = NOTEBOOK_DIR / "Output_experiemnt_RAG_v1_Dmptools"

# Optional: timestamped run folder so outputs never overwrite
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = NB_OUT_ROOT / f"run_{RUN_ID}"

NB_MD_DIR   = RUN_DIR / "md"
NB_DOCX_DIR = RUN_DIR / "docx"

for p in [NB_OUT_ROOT, RUN_DIR, NB_MD_DIR, NB_DOCX_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# ---------- YAML loader + path resolver ----------
def load_yaml_config(cfg_path: Path) -> dict:
    cfg_path = Path(cfg_path)
    if not cfg_path.exists():
        raise FileNotFoundError(f"Config YAML not found: {cfg_path}")
    with cfg_path.open("r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    if not isinstance(cfg, dict):
        raise ValueError(f"Config YAML must parse to a dict. Got: {type(cfg)}")
    return cfg


def resolve_from_root(project_root: Path, root_dir_value: str | Path) -> Path:
    """
    YAML root_dir:
      - "." means project root
      - relative paths are relative to project root
      - absolute paths are used as-is
    """
    p = Path(root_dir_value).expanduser()
    if p.is_absolute():
        return p.resolve()
    return (project_root / p).resolve()


def resolve_path(base: Path, rel_or_abs: str | Path | None) -> Path | None:
    """Resolve a path relative to `base` if not absolute. Keep None as None."""
    if rel_or_abs is None:
        return None
    p = Path(rel_or_abs).expanduser()
    if p.is_absolute():
        return p.resolve()
    return (base / p).resolve()


# ---------- Choose your YAML file here ----------
CONFIG_YAML = PROJECT_ROOT / "config" / "config.yaml"
cfg = load_yaml_config(CONFIG_YAML)

# ---------- Root dir from YAML ----------
ROOT_DIR = resolve_from_root(PROJECT_ROOT, cfg["root_dir"])

# ---------- Paths from YAML (READ-ONLY / pipeline assets) ----------
DATA_PDFS  = resolve_path(ROOT_DIR, cfg["paths"]["data_pdfs"])        # optional here, but kept for reference
INDEX_DIR  = resolve_path(ROOT_DIR, cfg["paths"]["index_dir"])        # <-- reuse pipeline index (read-only)
EXCEL_PATH = resolve_path(ROOT_DIR, cfg["paths"]["excel_path"])       # optional depending on your notebook

# Template (read-only)
TEMPLATE_MD = resolve_path(
    ROOT_DIR,
    cfg["paths"].get("template_md", "data/inputs/dmp-template.md")
)

# ---------- RAG params ----------
TOP_K = int(cfg["rag"]["retriever_top_k"])

# ---------- Models ----------
EMBED_MODEL = cfg["models"]["embedding_model"]
LLM_MODEL   = cfg["models"]["llm_name"]

EMBED_DEVICE       = cfg["models"]["embedding_device"]
EMBED_BATCH_SIZE   = int(cfg["models"]["embedding_batch_size"])
NORMALIZE_EMBEDS   = bool(cfg["models"]["normalize_embeddings"])
HF_CACHE_DIR       = resolve_path(ROOT_DIR, cfg["models"]["hf_cache_dir"])
LOCAL_FILES_ONLY   = bool(cfg["models"]["local_files_only"])
ALLOW_DL_IF_MISS   = bool(cfg["models"]["allow_download_if_missing"])

# ---------- Notebook-only outputs (always under noteboo_DMP_RAG/) ----------
OUTPUT_MD   = NB_MD_DIR / "generated_dmp.md"
OUTPUT_DOCX = NB_DOCX_DIR / "generated_dmp.docx"


# ---------- Helper functions ----------
def create_folder(folderpath: Path | str) -> None:
    Path(folderpath).mkdir(parents=True, exist_ok=True)

def save_md(folderpath: Path | str, filename: str, text: str) -> Path:
    create_folder(folderpath)
    out_path = Path(folderpath) / filename
    out_path.write_text(text, encoding="utf-8")
    print("Saved:", out_path)
    return out_path

def md_to_docs(md_filepath: Path | str, docx_folderpath: Path | str, docx_filename: str) -> Path:
    create_folder(docx_folderpath)
    out_path = Path(docx_folderpath) / docx_filename
    pypandoc.convert_file(str(md_filepath), "docx", outputfile=str(out_path))
    print("Converted:", out_path)
    return out_path

def clean_filename(name: str) -> str:
    """Remove illegal characters from filenames (Windows-safe)."""
    return re.sub(r'[\\/*?:"<>|]', "_", str(name)).strip()


# ---------- Sanity print ----------
print("STEP 1 ready (reuses pipeline index, notebook-local outputs)")
print(f"CONFIG_YAML : {CONFIG_YAML}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"ROOT_DIR    : {ROOT_DIR}")
print(f"NOTEBOOK_DIR: {NOTEBOOK_DIR}")
print(f"NB_OUT_ROOT : {NB_OUT_ROOT}")
print(f"RUN_DIR     : {RUN_DIR}")

print(f"INDEX_DIR (pipeline): {INDEX_DIR}")
print(f"OUTPUT_MD (notebook): {OUTPUT_MD}")
print(f"OUTPUT_DOCX(notebook): {OUTPUT_DOCX}")

print(f"DATA_PDFS   : {DATA_PDFS}")
print(f"EXCEL_PATH  : {EXCEL_PATH}")
print(f"TEMPLATE_MD : {TEMPLATE_MD}")

print(f"EMBED_MODEL : {EMBED_MODEL}")
print(f"LLM_MODEL   : {LLM_MODEL}")
print(f"TOP_K       : {TOP_K}")
print(f"EMBED_DEVICE: {EMBED_DEVICE} | BATCH: {EMBED_BATCH_SIZE} | NORMALIZE: {NORMALIZE_EMBEDS}")
print(f"HF_CACHE_DIR: {HF_CACHE_DIR} | local_files_only={LOCAL_FILES_ONLY} | allow_download_if_missing={ALLOW_DL_IF_MISS}")

STEP 1 ready (reuses pipeline index, notebook-local outputs)
CONFIG_YAML : c:\Users\Nahid\dmpchef\config\config.yaml
PROJECT_ROOT: c:\Users\Nahid\dmpchef
ROOT_DIR    : C:\Users\Nahid\dmpchef
NOTEBOOK_DIR: c:\Users\Nahid\dmpchef\notebook_DMP_RAG
NB_OUT_ROOT : c:\Users\Nahid\dmpchef\notebook_DMP_RAG\Output_experiemnt_RAG_v1_Dmptools
RUN_DIR     : c:\Users\Nahid\dmpchef\notebook_DMP_RAG\Output_experiemnt_RAG_v1_Dmptools\run_20260303_131330
INDEX_DIR (pipeline): C:\Users\Nahid\dmpchef\data\vector_db\DMPtools_db
OUTPUT_MD (notebook): c:\Users\Nahid\dmpchef\notebook_DMP_RAG\Output_experiemnt_RAG_v1_Dmptools\run_20260303_131330\md\generated_dmp.md
OUTPUT_DOCX(notebook): c:\Users\Nahid\dmpchef\notebook_DMP_RAG\Output_experiemnt_RAG_v1_Dmptools\run_20260303_131330\docx\generated_dmp.docx
DATA_PDFS   : C:\Users\Nahid\dmpchef\data\data_ingestion\DMPtools
EXCEL_PATH  : C:\Users\Nahid\dmpchef\data\inputs\inputs.xlsx
TEMPLATE_MD : C:\Users\Nahid\dmpchef\data\inputs\dmp-template.md
EMBED_MODEL : sent

In [21]:
# ============================================
# STEP 2 — Load pipeline FAISS index (READ-ONLY) and create retriever
# ============================================
from pathlib import Path
from langchain_community.vectorstores import FAISS

# Prefer new HuggingFace integration if installed; fallback to langchain_community
try:
    from langchain_huggingface import HuggingFaceEmbeddings  # type: ignore
    _EMB_BACKEND = "langchain_huggingface"
except Exception:
    from langchain_community.embeddings import HuggingFaceEmbeddings  # type: ignore
    _EMB_BACKEND = "langchain_community"

import torch
import os

# Ensure HF uses the same cache directory you configured (very important in notebooks)
# This helps avoid "couldn't find them in the cached files" surprises.
if HF_CACHE_DIR is not None:
    os.environ.setdefault("HF_HOME", str(HF_CACHE_DIR))

def _pick_device(requested: str) -> str:
    req = (requested or "auto").lower().strip()
    if req in ("auto", "cuda"):
        return "cuda" if torch.cuda.is_available() else "cpu"
    return "cpu"

device = _pick_device(EMBED_DEVICE)

def _make_embeddings(local_only: bool):
    return HuggingFaceEmbeddings(
        model_name=EMBED_MODEL,
        cache_folder=str(HF_CACHE_DIR) if HF_CACHE_DIR is not None else None,
        model_kwargs={
            "device": device,
            "local_files_only": bool(local_only),
        },
        encode_kwargs={
            "batch_size": int(EMBED_BATCH_SIZE),
            "normalize_embeddings": bool(NORMALIZE_EMBEDS),
        },
    )

# 1) Try per YAML: local_files_only
try:
    embeddings = _make_embeddings(local_only=LOCAL_FILES_ONLY)
except Exception as e1:
    # 2) If offline cache miss but allowed to download, retry online
    if LOCAL_FILES_ONLY and ALLOW_DL_IF_MISS:
        print("Embeddings not found in cache; retrying with download enabled...")
        embeddings = _make_embeddings(local_only=False)
    else:
        raise

index_dir = Path(INDEX_DIR)
faiss_path = index_dir / "index.faiss"
pkl_path   = index_dir / "index.pkl"

if not (faiss_path.exists() and pkl_path.exists()):
    raise FileNotFoundError(
        "FAISS index files not found.\n"
        f"Expected:\n- {faiss_path}\n- {pkl_path}\n"
        "Run your pipeline build_index.py or fix config.paths.index_dir."
    )

print("Loading FAISS index (read-only) from:", index_dir)
vectorstore = FAISS.load_local(
    str(index_dir),
    embeddings,
    allow_dangerous_deserialization=True
)

retriever = vectorstore.as_retriever(search_kwargs={"k": int(TOP_K)})

print(
    "Retriever ready",
    f"top_k={TOP_K}",
    f"embed_model={EMBED_MODEL}",
    f"device={device}",
    f"backend={_EMB_BACKEND}",
    sep=" | "
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1410.17it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading FAISS index (read-only) from: C:\Users\Nahid\dmpchef\data\vector_db\DMPtools_db
Retriever ready | top_k=6 | embed_model=sentence-transformers/all-MiniLM-L6-v2 | device=cuda | backend=langchain_huggingface


In [22]:
# ============================================
# STEP 3 — Load Excel + Template, Build Few-shot, and Build RAG Chain
# ============================================
import re
import pandas as pd
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.llms import Ollama

# ---- Guard: Step 2 must run first ----
if "retriever" not in globals():
    raise RuntimeError("`retriever` is not defined. Run STEP 2 (load FAISS index) before STEP 3.")

# --- Load Excel file ---
if EXCEL_PATH is None or not EXCEL_PATH.exists():
    raise FileNotFoundError(f"Excel file not found: {EXCEL_PATH}")

df = pd.read_excel(EXCEL_PATH)
df.columns = df.columns.str.strip().str.lower()
df = df.fillna("")
print(f"Excel loaded successfully: {len(df)} rows")

# --- Load Markdown Template ---
if TEMPLATE_MD is None or not TEMPLATE_MD.exists():
    raise FileNotFoundError(f"Template file not found: {TEMPLATE_MD}")

dmp_template_text = TEMPLATE_MD.read_text(encoding="utf-8")
print("DMP Markdown template loaded.")


# ============================================
# Build RAG chain (RAG grounding)
# ============================================
def build_rag_chain(retriever, llm_model=LLM_MODEL, n_few_shot: int = 3):
    llm = Ollama(model=llm_model)
    parser = StrOutputParser()

    def format_docs(docs):
        if not docs:
            return ""
        formatted = []
        for d in docs:
            page = d.metadata.get("page", d.metadata.get("page_number", ""))
            source = d.metadata.get("source", d.metadata.get("file_path", ""))
            page_disp = (page + 1) if isinstance(page, int) else page
            formatted.append(f"[Page {page_disp}] {source}\n{(d.page_content or '').strip()}")
        return "\n\n".join(formatted)

    prompt_template = """You are an expert biomedical data steward and grant writer.
Create a high-quality NIH Data Management and Sharing Plan (DMSP) based on the retrieved NIH context and the user's query.

---- Context from NIH Repository (grounding) ----
{context}

---- Question ----
{question}

Rules:
- Use NIH context when relevant; do NOT invent policy details.
- If a specific policy detail is not supported by the provided context, write: "Not specified in provided NIH context."
- Follow the NIH template structure and keep section titles unchanged when the template is provided.
"""
    prompt = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

    rag_chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough(),
        }
        | prompt
        | llm
        | parser
    )

    print(f"RAG chain initialized with model: {llm_model}")
    return rag_chain


rag_chain = build_rag_chain(retriever, n_few_shot=3)
print("RAG chain ready for generation.")

Excel loaded successfully: 26 rows
DMP Markdown template loaded.
RAG chain initialized with model: llama3.3:latest
RAG chain ready for generation.


C:\Users\Nahid\AppData\Local\Temp\ipykernel_41104\281723273.py:36: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model=llm_model)


In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_template = """You are an expert biomedical data steward and grant writer.
Create a high-quality NIH Data Management and Sharing Plan (DMSP) based on the retrieved NIH context and the user's query.

---- Context from NIH Repository (grounding) ----
{context}

---- Question ----
{question}

Rules:
- Use NIH context when relevant; do NOT invent policy details.
- If a specific policy detail is not supported by the provided context, write: "Not specified in provided NIH context."
- Follow the NIH template structure and keep section titles unchanged when the template is provided.
"""
prompt = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

def format_docs(docs):
    if not docs:
        return ""
    formatted = []
    for d in docs:
        page = d.metadata.get("page", d.metadata.get("page_number", ""))
        source = d.metadata.get("source", d.metadata.get("file_path", ""))
        page_disp = (page + 1) if isinstance(page, int) else page
        formatted.append(f"[Page {page_disp}] {source}\n{(d.page_content or '').strip()}")
    return "\n\n".join(formatted)

# ---- Build question from first Excel row (same as STEP 4) ----
row = df.iloc[0]
title = str(row.get("title", "")).strip()

element_cols = [c for c in df.columns if str(c).startswith("element")]
element_texts = []
for col in element_cols:
    val = str(row.get(col, "")).strip()
    if val:
        element_texts.append(f"{col.upper()}: {val}")
query_data = "\n".join(element_texts).strip()

question = f"""
Create a complete NIH Data Management and Sharing Plan (DMSP) for the project titled: "{title}".

Use the NIH DMSP Markdown template below and DO NOT change section titles.

Project background / proposal details:
{query_data}

NIH DMSP Markdown template:
{dmp_template_text}
""".strip()


docs = retriever.invoke(question)   # List[Document]
context = format_docs(docs)

final_prompt = prompt.format(context=context, question=question)

print("\n========== FINAL PROMPT (SENT TO LLM) ==========\n")
print(final_prompt)

print("\nContains 'labname'?", "labname" in final_prompt.lower())
print("Contains 'university x'?", "university x" in final_prompt.lower())


========== FINAL PROMPT (SENT TO LLM) ==========

You are an expert biomedical data steward and grant writer.
Create a high-quality NIH Data Management and Sharing Plan (DMSP) based on the retrieved NIH context and the user's query.

---- Context from NIH Repository (grounding) ----
[Page 4] C:\Users\Nahid\dmpchef\data\data_ingestion\DMPtools\dmptool_dmp_0060.pdf
Imaging data will be deposited into NCI’s Imaging Data Commons or the Brain Image Library. All
other data described above in the “data to be shared” section will be deposited into the DANDI
Archive.
How scientific data will be findable and identifiable: Describe how the
scientific data will be findable and identifiable, i.e., via a persistent unique
identifier or other standard indexing tools.
For electrophysiolgical data, recordings will be organized by Aim and subaim into dated folders. A
spreadsheet will be provided with all key information (Subject #, Sex, Treatment, etc).
For imaging data, pictures will be preserved in r